# Text Emotion Classification and Telegram Prototype

Project-ready paths and setup are added to the supplied experiment. The analytical methods are preserved; this copy has no saved outputs. See `../originals/text.sanitized.ipynb` for the historical run and `../docs/REPRODUCIBILITY.md` for changes, dependencies, and limitations.

Run the setup cell first, then continue in order. Download datasets separately. Saved model weights are not included.

**Research prototype:** the supplied Transformer has a batch/sequence layout issue. Correct that issue and retrain before claiming text accuracy or using the bot. The bot is opt-in and requires `RUN_TELEGRAM_BOT=1` and `TELEGRAM_BOT_TOKEN`.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'data'
ARTIFACT_DIR = REPO_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
SPEECH_DATA_DIR = Path(os.environ.get('SPEECH_DATA_DIR', str(DATA_DIR / 'speech-emotion-recognition-en')))
TEXT_DATA_PATH = Path(os.environ.get('TEXT_DATA_PATH', str(DATA_DIR / 'text/tweet_emotions.csv')))
GLOVE_PATH = Path(os.environ.get('GLOVE_PATH', str(DATA_DIR / 'glove/glove.6B.50d.txt')))
SPEECH_SPLIT_DIR = ARTIFACT_DIR / 'speech_splits'
SPEECH_SPLIT_DIR.mkdir(exist_ok=True)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt # plotting
from wordcloud import WordCloud, STOPWORDS
from collections import defaultdict
import numpy as np # linear algebra
import os # accessing directory structure
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv(TEXT_DATA_PATH, delimiter=',')
df.head()


In [ ]:
def basic_eda(df, row_limit=5, list_elements_limit=10):
    ### rows and columns
    print('Info : There are {} columns in the dataset'.format(df.shape[1]))
    print('Info : There are {} rows in the dataset'.format(df.shape[0]))
    
    print("==================================================")
    
    ## data types
    print("\nData type information of different columns")
    dtypes_df = pd.DataFrame(df.dtypes).reset_index().rename(columns={0:'dtype', 'index':'column_name'})
    cat_df = dtypes_df[dtypes_df['dtype']=='object']
    num_df = dtypes_df[dtypes_df['dtype']!='object']
    print('Info : There are {} categorical columns'.format(len(cat_df)))
    print('Info : There are {} numerical columns'.format(len(dtypes_df)-len(cat_df)))
    
    if list_elements_limit >= len(cat_df):
        print("Categorical columns : ", list(cat_df['column_name']))
    else:
        print("Categorical columns : ", list(cat_df['column_name'])[:list_elements_limit])
        
    if list_elements_limit >= len(num_df):
        print("Numerical columns : ", list(num_df['column_name']))
    else:
        print("Numerical columns : ", list(num_df['column_name'])[:list_elements_limit])
    
    #dtypes_df['dtype'].value_counts().plot.bar()
    display(dtypes_df.head(row_limit))
    
    print("==================================================")
    print("\nDescription of numerical variables")
    
    #### Describibg numerical columns
    desc_df_num = df[list(num_df['column_name'])].describe().T.reset_index().rename(columns={'index':'column_name'})
    display(desc_df_num.head(row_limit))
    
    print("==================================================")
    print("\nDescription of categorical variables")
    
    desc_df_cat = df[list(cat_df['column_name'])].describe().T.reset_index().rename(columns={'index':'column_name'})
    display(desc_df_cat.head(row_limit))
    
    return


In [ ]:
basic_eda(df)


In [ ]:
col = 'sentiment'
fig, (ax1, ax2)  = plt.subplots(nrows=1, ncols=2, figsize=(12,8))
explode = list((np.array(list(df[col].dropna().value_counts()))/sum(list(df[col].dropna().value_counts())))[::-1])[:10]
labels = list(df[col].dropna().unique())[:10]
sizes = df[col].value_counts()[:10]
#ax.pie(sizes, explode=explode, colors=bo, startangle=60, labels=labels,autopct='%1.0f%%', pctdistance=0.9)
ax2.pie(sizes,  explode=explode, startangle=60, labels=labels,autopct='%1.0f%%', pctdistance=0.9)
ax2.add_artist(plt.Circle((0,0),0.6,fc='white'))
sns.countplot(y =col, data = df, ax=ax1)
ax1.set_title("Count of each emotion")
ax2.set_title("Percentage of each emotion")
plt.show()


In [ ]:
# Dictionary to map 13 original labels into 6 categories
label_mapping = {
    'empty': 'neutral',
    'sadness': 'worry',
    'enthusiasm': 'surprise',
    'neutral': 'neutral',
    'worry': 'worry',
    'surprise': 'surprise',
    'love': 'love',
    'fun': 'happiness',
    'hate': 'anger',
    'happiness': 'happiness',
    'boredom': 'neutral',
    'relief': 'surprise',
    'anger': 'anger'
}

# Apply the mapping to the label column
df['sentiment'] = df['sentiment'].map(label_mapping)
df.head()


In [ ]:
# Install dependencies from the repository requirements files.
# Install dependencies from the repository requirements files.


In [ ]:
# Install dependencies from the repository requirements files.


In [ ]:
# Data preparation and text-preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import nltk
import inflect
import contractions
from bs4 import BeautifulSoup
import re, string, unicodedata
from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import LancasterStemmer, WordNetLemmatizer


In [ ]:
def text_preprocessing_platform(df, text_col, remove_stopwords=True):
    
    ## Define functions for individual steps
    # First function is used to denoise text
    def denoise_text(text):
        # Strip html if any. For ex. removing <html>, <p> tags
        soup = BeautifulSoup(text, "html.parser")
        text = soup.get_text()
        # Replace contractions in the text. For ex. didn't -> did not
        text = contractions.fix(text)
        return text
    
    ## Next step is text-normalization
    
    # Text normalization includes many steps.
    
    # Each function below serves a step.
    
    
    def remove_non_ascii(words):
        """Remove non-ASCII characters from list of tokenized words"""
        new_words = []
        for word in words:
            new_word = unicodedata.normalize('NFKD', word).encode('ascii', 'ignore').decode('utf-8', 'ignore')
            new_words.append(new_word)
        return new_words
    
    
    def to_lowercase(words):
        """Convert all characters to lowercase from list of tokenized words"""
        new_words = []
        for word in words:
            new_word = word.lower()
            new_words.append(new_word)
        return new_words
    
    
    def remove_punctuation(words):
        """Remove punctuation from list of tokenized words"""
        new_words = []
        for word in words:
            new_word = re.sub(r'[^\w\s]', '', word)
            if new_word != '':
                new_words.append(new_word)
        return new_words
    
    
    def replace_numbers(words):
        """Replace all interger occurrences in list of tokenized words with textual representation"""
        p = inflect.engine()
        new_words = []
        for word in words:
            if word.isdigit():
                new_word = p.number_to_words(word)
                new_words.append(new_word)
            else:
                new_words.append(word)
        return new_words
    
    
    def remove_stopwords(words):
        """Remove stop words from list of tokenized words"""
        new_words = []
        for word in words:
            if word not in stopwords.words('english'):
                new_words.append(word)
        return new_words
    
    
    def stem_words(words):
        """Stem words in list of tokenized words"""
        stemmer = LancasterStemmer()
        stems = []
        for word in words:
            stem = stemmer.stem(word)
            stems.append(stem)
        return stems
    
    
    def lemmatize_verbs(words):
        """Lemmatize verbs in list of tokenized words"""
        lemmatizer = WordNetLemmatizer()
        lemmas = []
        for word in words:
            lemma = lemmatizer.lemmatize(word, pos='v')
            lemmas.append(lemma)
        return lemmas
    
    
    ### A wrap-up function for normalization
    def normalize_text(words, remove_stopwords):
        words = remove_non_ascii(words)
        words = to_lowercase(words)
        words = remove_punctuation(words)
        words = replace_numbers(words)
        if remove_stopwords:
            words = remove_stopwords(words)
        #words = stem_words(words)
        words = lemmatize_verbs(words)
        return words
    
    # All above functions work on word tokens we need a tokenizer
    
    # Tokenize tweet into words
    def tokenize(text):
        return nltk.word_tokenize(text)
    
    
    # A overall wrap-up function
    def text_prepare(text):
        text = denoise_text(text)
        text = ' '.join([x for x in normalize_text(tokenize(text), remove_stopwords)])
        return text
    
    # run every-step
    df[text_col] = [text_prepare(x) for x in df[text_col]]
    
    
    # return processed df
    return df


In [ ]:
print("Before Text Preprocessing")
display(df.head()[['content']])
df = text_preprocessing_platform(df, 'content', remove_stopwords=False)
print("After Text Preprocessing")
display(df.head()[['content']])


In [ ]:
def print_word_cloud(df, sentiment):

    print("Word cloud of most frequent words for the sentiment : {}".format(sentiment))

    temp_df = df[df['sentiment']==sentiment]
    print("Number of Rows : ", len(temp_df))

    corpus = ''
    for text in temp_df.content:
        text = str(text)
        corpus += text
        
    total = 0
    count = defaultdict(lambda: 0)
    for word in corpus.split(" "):
        total += 1
        count[word] += 1
        
    top20pairs = sorted(count.items(), key=lambda kv: kv[1], reverse=True)[:20]
    top20words = [i[0] for i in top20pairs]
    top20freq = [i[1] for i in top20pairs]
    
    xs = np.arange(len(top20words))
    width = 0.5

    fig = plt.figure(figsize=(10,6))                                                               
    ax = fig.gca()  #get current axes
    ax.bar(xs, top20freq, width, align='center')

    ax.set_xticks(xs)
    ax.set_xticklabels(top20words)
    plt.xticks(rotation=45)
    
    
    stopwords = set(STOPWORDS)
    # lower max_font_size, change the maximum number of word and lighten the background:
    wordcloud = WordCloud(max_font_size=50, max_words=50,stopwords=stopwords, background_color="white").generate(corpus)
    plt.figure(figsize = (12, 12), facecolor = None)
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()


In [ ]:
print_word_cloud(df, 'worry')


In [ ]:
print_word_cloud(df, 'happiness')


In [ ]:
# Install dependencies from the repository requirements files.
# Install dependencies from the repository requirements files.
# Install dependencies from the repository requirements files.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import os
import logging


In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define your TransformerTextClassifier class (same as in your code)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class TransformerTextClassifier(nn.Module):
    def __init__(self, word_index, embeddings_index, nclasses, max_seq_len=500, embedding_dim=50, dropout=0.2, num_heads=5, num_layers=2):
        super(TransformerTextClassifier, self).__init__()
        self.embedding = nn.Embedding(len(word_index) + 1, embedding_dim)
        self._init_embeddings(embeddings_index, word_index)
        self.pos_encoder = PositionalEncoding(embedding_dim, dropout, max_seq_len)
        encoder_layers = nn.TransformerEncoderLayer(embedding_dim, num_heads, dim_feedforward=256, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(embedding_dim, 25)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(25, nclasses)

    def _init_embeddings(self, embeddings_index, word_index):
        embedding_matrix = np.zeros((len(word_index) + 1, 50))
        for word, i in word_index.items():
            embedding_vector = embeddings_index.get(word)
            if embedding_vector is not None:
                embedding_matrix[i] = embedding_vector
        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.embedding.weight.requires_grad = False

    def forward(self, x):
        x = self.embedding(x) * np.sqrt(50)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [ ]:
###Utility

def get_eval_report(labels, preds):
    mcc = matthews_corrcoef(labels, preds)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    precision = (tp)/(tp+fp)
    recall = (tp)/(tp+fn)
    f1 = (2*(precision*recall))/(precision+recall)
    return {
        "mcc": mcc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "pricision" : precision,
        "recall" : recall,
        "F1" : f1,
        "accuracy": (tp+tn)/(tp+tn+fp+fn)
    }

def compute_metrics(labels, preds):
    assert len(preds) == len(labels)
    return get_eval_report(labels, preds)

def plot_graphs(history, string):
  plt.plot(history.history[string])
  plt.plot(history.history['val_'+string], '')
  plt.xlabel("Epochs")
  plt.ylabel(string)
  plt.legend([string, 'val_'+string])
  plt.show()

def class_balance(df, target):
  cls = df[target].value_counts()
  cls.plot(kind='bar')
  plt.show()


In [ ]:
# Function to load GloVe embeddings
def load_glove_embeddings(glove_path):
    embeddings_index = {}
    with open(glove_path, encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    return embeddings_index


In [ ]:
# Function to tokenize text
def loadData_Tokenizer(X_train, X_test, glove_path, max_seq_len=500):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(X_train)
    word_index = tokenizer.word_index

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_seq_len)
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_seq_len)

    embeddings_index = load_glove_embeddings(glove_path)
    return X_train_pad, X_test_pad, word_index, embeddings_index, tokenizer


In [ ]:
# Function to preprocess and classify text
def classify_text(text, model, tokenizer, max_seq_len, device):
    # Tokenize and pad the input text
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_seq_len)
    input_tensor = torch.tensor(padded, dtype=torch.long).to(device)

    # Model inference
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        pred = torch.argmax(output, dim=1).cpu().numpy()[0]
    return pred

# Function to convert voice to text
def voice_to_text(audio_file):
    recognizer = sr.Recognizer()
    audio = AudioSegment.from_file(audio_file)
    audio.export("temp.wav", format="wav")
    
    with sr.AudioFile("temp.wav") as source:
        audio_data = recognizer.record(source)
        try:
            text = recognizer.recognize_google(audio_data)
            return text
        except sr.UnknownValueError:
            return "Could not understand the audio"
        except sr.RequestError:
            return "Speech recognition service unavailable"
        finally:
            if os.path.exists("temp.wav"):
                os.remove("temp.wav")


In [ ]:
df = df[['content', 'sentiment']]
print("Train DataFrame")
df = df.sample(frac=1).reset_index(drop=True)
display(df.head(40))


In [ ]:
# Label encoding target column
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])
df.head(40)


In [ ]:
# Model Building
from tensorflow.keras.layers import Dropout, Dense, GRU, Embedding, LSTM, Bidirectional, TimeDistributed, Flatten
from tensorflow.keras.models import Sequential
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import matthews_corrcoef, confusion_matrix
from sklearn import metrics
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

# Logging
import logging
logging.basicConfig(level=logging.INFO)


In [ ]:
import pickle

X = df['content']
y = df['sentiment']
glove_path = str(GLOVE_PATH)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print("Generating Glove Embeddings...")
X_train_Glove, X_test_Glove, word_index, embeddings_index, tokenizer = loadData_Tokenizer(X_train, X_test, glove_path)

# Save the tokenizer
with open(str(ARTIFACT_DIR / "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

# Continue with the rest of your training code...


In [ ]:
X_train_tensor = torch.tensor(X_train_Glove, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_Glove, dtype=torch.long)
y_train_tensor = torch.tensor(np.array(y_train), dtype=torch.long)
y_test_tensor = torch.tensor(np.array(y_test), dtype=torch.long)


In [ ]:
model = TransformerTextClassifier(
    word_index=word_index,
    embeddings_index=embeddings_index,
    nclasses=6,                 # your number of classes
    max_seq_len=500,
    embedding_dim=50,
    dropout=0.2,
    num_heads=10,
    num_layers=2
)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import numpy as np
import os


In [ ]:
def train_model(model, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor,
                batch_size=32, epochs=100, lr=0.0001, weight_decay=0.01,
                patience=5, step_size=5, gamma=0.1, save_path=str(ARTIFACT_DIR / "text_best_model.pt")):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    best_val_loss = np.inf
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        all_preds = []
        all_labels = []

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(all_labels, all_preds)
        avg_train_loss = train_loss / len(train_loader)

        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)

        print(f"Epoch {epoch+1}: "
              f"Train Loss = {avg_train_loss:.4f}, Train Acc = {train_acc:.4f} | "
              f"Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")

        scheduler.step()

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✓ Saved new best model at epoch {epoch+1} (Val Loss: {val_loss:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(save_path))
    return model


In [ ]:
def evaluate_model(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    return avg_val_loss, val_acc


In [ ]:
train_model(model , X_train_tensor , y_train_tensor , X_test_tensor , y_test_tensor)


In [ ]:
# Initialize the bot
def main():
    # Your Telegram bot token
    TELEGRAM_TOKEN = os.environ["TELEGRAM_BOT_TOKEN"]

    # Paths and configurations
    glove_path = str(GLOVE_PATH)  # Adjust path if necessary
    model_path = str(ARTIFACT_DIR / "text_best_model.pt")
    tokenizer_path = str(ARTIFACT_DIR / "tokenizer.pkl")  # Path to the saved tokenizer
    max_seq_len = 500
    nclasses = 6  # Adjust based on your number of sentiment classes
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load the saved tokenizer
    with open(tokenizer_path, "rb") as f:
        tokenizer = pickle.load(f)
    word_index = tokenizer.word_index

    # Load GloVe embeddings
    embeddings_index = load_glove_embeddings(glove_path)

    # Load the model
    model = TransformerTextClassifier(word_index, embeddings_index, nclasses, max_seq_len=max_seq_len, embedding_dim=50)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)

    # Define sentiment labels (adjust based on your LabelEncoder mapping)
    sentiment_labels = {0: "anger", 1: "happiness", 2: "love", 3: "neutral", 4: "surprise", 5: "worry"}

    # Set up the Telegram bot
    updater = Updater(TELEGRAM_TOKEN, use_context=True)
    dp = updater.dispatcher

    # Handler for /start command
    def start(update, context):
        update.message.reply_text("Hello! Send me a text or voice message, and I'll classify the sentiment.")

    # Handler for text messages
    def handle_text(update, context):
        text = update.message.text
        pred = classify_text(text, model, tokenizer, max_seq_len, device)
        sentiment = sentiment_labels.get(pred, "Unknown")
        update.message.reply_text(f"Sentiment: {sentiment}")

    # Handler for voice messages
    def handle_voice(update, context):
        voice = update.message.voice
        file = context.bot.get_file(voice.file_id)
        file.download("voice_message.ogg")

        # Convert voice to text
        text = voice_to_text("voice_message.ogg")
        if "Could not understand" in text or "service unavailable" in text:
            update.message.reply_text(text)
        else:
            # Classify the text
            pred = classify_text(text, model, tokenizer, max_seq_len, device)
            sentiment = sentiment_labels.get(pred, "Unknown")
            update.message.reply_text(f"Transcribed Text: {text}\nSentiment: {sentiment}")

        # Clean up
        if os.path.exists("voice_message.ogg"):
            os.remove("voice_message.ogg")

    # Add handlers
    dp.add_handler(CommandHandler("start", start))
    dp.add_handler(MessageHandler(Filters.text & ~Filters.command, handle_text))
    dp.add_handler(MessageHandler(Filters.voice, handle_voice))

    # Start the bot
    updater.start_polling()
    updater.idle()

if __name__ == "__main__" and os.environ.get("RUN_TELEGRAM_BOT") == "1":
    import telegram
    from telegram.ext import Updater, CommandHandler, MessageHandler, Filters
    import speech_recognition as sr
    from pydub import AudioSegment
    main()
